In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import lightkurve as lk
import glob
import re
from astroquery import vizier
import pandas as pd
from astroquery.mast import Catalogs
from astropy.table import Table

In [2]:
final_tic_converted = ['257436461', '435339847', '337607207', '39936432', '392469577', '266014079', '14570099', '179032688', '376938120', '187309502', '333605244', '366631954', '307733361', '332022997', '238608004', '376939759', '418761354', '293460551', '12822545', '187960878', '440687723', '175241633', '7059054', '297146957', '38258419', '2621212', '175265494', '293375721', '175194958', '380884458', '187273811', '175290532', '26078330', '195193025', '14635562', '677945', '115072772', '12181371', '741596', '146799150', '90090343', '294319820', '458686847', '423358488', '125060509', '175262071', '113979956', '376936928', '135043332', '14111411', '2521105', '332024125', '434226736', '11023038', '15756231', '12770870', '86130229', '197246627', '422349422', '441907126', '10195089', '335102224',  '396972464', '184914317', '49652731', '301258470', '68504570', '446672575', '291090078', '49040478', '4610830', '428820090', '281731203', '301610040', '175193677', '363445338', '335322931', '363445121', '27039476', '301235044', '323687123', '345063080', '380619414', '436575927', '38064757', '277869696', '397022282', '38087018', '18310799', '281886199', '5882269', '150096001', '456945304', '49462779', '50171060', '388804061', '422487869', '82050863', '363444743', '173090902', '443616612', '392817207', '432247315', '903075188', '61312205']


final_periods = [33.122, 20.686, 8.733, 29.429, 17.988, 9.007, 30.178, 24.824, 33.119, 20.866, 26.33, 24.244, 4.263, 9.117, 18.253, 40.0, 33.122, 30.19, 16.414, 16.826, 23.532, 11.113, 10.187, 35.639, 21.761, 15.821, 10.788, 16.23, 17.125, 30.5, 13.625, 40.0, 8.891, 24.244, 33.181, 12.03, 26.855, 26.812, 29.005, 12.677, 23.688, 23.065, 9.179, 14.466, 13.047, 11.457, 35.822, 40.0, 7.056, 25.266, 17.997, 20.239, 1.878, 17.552, 2.844, 17.263, 9.557, 36.182, 28.418, 14.436, 40.0, 40.0, 15.393, 21.954, 23.218, 12.356, 12.226, 7.849, 14.823, 6.28, 12.356, 9.655, 18.794, 11.428, 8.33, 7.698, 15.016, 8.517, 18.132, 7.044, 4.338, 14.053, 6.562, 28.531, 13.341, 7.999, 21.062, 21.062, 14.768, 14.294, 8.854, 36.347, 13.289, 19.057, 17.077, 36.367, 23.366, 36.367, 14.294, 20.009, 9.763, 6.637, 10.868, 21.062, 36.279]

In [3]:
def get_coords_from_tic(tic_ids):
    """
    Queries MAST to get RA and Dec for a list of TIC IDs.
    """
    # Ensure IDs are strings for the query
    tic_ids = [str(tid) for tid in tic_ids]
    
    print(f"Querying MAST for {len(tic_ids)} IDs...")
    
    # Query the TIC catalog
    # 'ID' in query_criteria refers to the TIC ID
    result_table = Catalogs.query_criteria(catalog="TIC", ID=tic_ids)
    
    # Convert to pandas for easier manipulation
    df_coords = result_table.to_pandas()
    
    # Select only the essential columns
    # 'ra' and 'dec' are J2000 coordinates
    df_coords = df_coords[['ID', 'ra', 'dec']]
    df_coords.columns = ['TIC_ID', 'RA', 'Dec']
    
    return df_coords

coord_df = get_coords_from_tic(final_tic_converted)

print("\nResults:")
print(coord_df)

# To use these in your existing cross-match dictionary:
# ra_dict = dict(zip(coord_df['TIC_ID'].astype(int), coord_df['RA']))
# dec_dict = dict(zip(coord_df['TIC_ID'].astype(int), coord_df['Dec']))

Querying MAST for 105 IDs...

Results:
        TIC_ID          RA        Dec
0    125060509  293.560726 -23.131101
1    175194958  129.636817  19.773775
2    293375721  131.266611  13.549830
3    175193677  129.689668  23.685906
4     10195089  288.026922 -21.007641
..         ...         ...        ...
100   18310799   67.412474  22.882721
101  436575927   71.109826  16.518680
102   14570099  125.987074  22.663038
103  363444743  169.483616   5.988393
104  293460551  132.009733  16.901863

[105 rows x 3 columns]


In [4]:
mega_list =pd.read_csv( 'ASTR502_Mega_Target_List.csv')
mega_list['tic_id'] = mega_list['tic_id'].str.replace('TIC ', '', regex=False).astype('Int64')

In [5]:
jayasinghe = Table.read('/Users/iansterrett/Downloads/jayasinghe2018/catv2021.dat', readme='https://cdsarc.cds.unistra.fr/ftp/II/366/ReadMe', format='ascii.cds').to_pandas()

In [ ]:
mega_with_jayasinghe = mega_list.merge(jayasinghe[['TIC', 'Per']], left_on='tic_id', right_on='TIC').drop_duplicates(subset='tic_id')

In [ ]:
print(mega_with_jayaisinghe)

             pl_name     hostname                   gaia_dr3_id  \
0           KELT-1 b       KELT-1  Gaia DR3 2881784280929695360   
1         TOI-1224 c     TOI-1224  Gaia DR3 4620009665047355520   
3         TOI-6109 b     TOI-6109   Gaia DR3 241035596174886016   
5            K2-89 b        K2-89    Gaia DR3 63206117414110848   
6        V1298 Tau b    V1298 Tau    Gaia DR3 51886335968692480   
10       TOI-3353.01     TOI-3353  Gaia DR3 5212899427468919296   
11         TOI-500 b      TOI-500  Gaia DR3 5509620021956148736   
12       TOI-5082.01     TOI-5082  Gaia DR3 3368214650329888512   
13        HD 63433 b     HD 63433   Gaia DR3 875071278432954240   
16        HD 73583 b     HD 73583  Gaia DR3 5746824674801810816   
18          G 9-40 b       G 9-40   Gaia DR3 684992690384102528   
19       HIP 67522 b    HIP 67522  Gaia DR3 6113920619134019456   
21        TOI-1860 b     TOI-1860  Gaia DR3 1620024491110370688   
22          K2-240 c       K2-240  Gaia DR3 625762571943098201

In [12]:
print(len(mega_with_jayaisinghe))

35


In [ ]:
mega_list =pd.read_csv( 'ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
schochet2025 = pd.read_csv('/Users/iansterrett/Downloads/reportable.csv')
print(schochet2025.columns)
mega_with_schochet = mega_list.merge(schochet2025[['dr3_source_id', 'period']], left_on='gaia_dr3_id', right_on='dr3_source_id').drop_duplicates(subset='gaia_dr3_id')
print(len(mega_with_schochet))

Index(['Unnamed: 0', 'asas_sn_id', 'period', 'sigma', 'dr3_source_id',
       'dr2_source_id', 'edr3_source_id', 'KIC', 'tic_id', 'ra_rad', 'dec_rad',
       'parallax', 'abs_g_mag', 'Tmag', 'phot_g_mean_mag', 'phot_bp_mean_mag',
       'phot_rp_mean_mag', 'radius_val', 'lum_val', 'ruwe', 'vbroad',
       'vbroad_error', 'catwise_w1', 'catwise_w2', 'mh_xgboost',
       'teff_xgboost', 'logg_xgboost', 'in_training', 'in_xgboost_training',
       'simulation_number'],
      dtype='object')
39


In [ ]:
print(mega_with_schochet)

              pl_name         hostname          gaia_dr3_id  \
0          HIP 65 A b         HIP 65 A  4923860051276772608   
1           GJ 3090 b          GJ 3090  4933912198895807104   
2           WASP-50 b          WASP-50  5160557726183065984   
3          TOI-2427 b         TOI-2427  5055663973297050624   
4           TOI-411 c         HD 22946  4848767461548943104   
7   IRAS 04125+2902 b  IRAS 04125+2902   164800235906366976   
8            NGTS-6 b           NGTS-6  4875693023844840448   
9          TOI-4364 b         TOI-4364  3210444215030339584   
10          HATS-43 b          HATS-43  2905983466705521792   
11          TOI-431 b          TOI-431  2908664557091200768   
13          WASP-19 b          WASP-19  5411736896952029568   
14           K2-159 b           K2-159  3600851450836925312   
15         TOI-1811 b         TOI-1811  3962403923821595264   
16           K2-128 b           K2-128  3618386664139093504   
17         TOI-1855 b         TOI-1855  124760371934570

In [ ]:
mega_list =pd.read_csv( 'ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
long = Table.read('/Users/iansterrett/Downloads/long2023/table5.dat', readme='https://cdsarc.cds.unistra.fr/ftp/J/ApJS/268/30/ReadMe', format='ascii.cds').to_pandas()
print('catalogue read in done')
print(long.columns)
mega_with_long = mega_list.merge(long[['Gaia', 'PRot']], left_on='gaia_dr3_id', right_on='Gaia').drop_duplicates(subset='gaia_dr3_id')
print(len(mega_with_long))

catalogue read in done
Index(['Cl', 'Gaia', 'RAdeg', 'DEdeg', 'RUWE', 'Prob', 'BinPhot', 'BinRUWE',
       'r_SB', 'SB', 'PRot', 'e_PRot', 'PSNR', 'r_PRot'],
      dtype='object')
18


In [ ]:
mega_list =pd.read_csv( 'ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
van_lane = Table.read('https://content.cld.iop.org/journals/0004-637X/986/1/59/ revision1/apjadcd73t1_mrt.txt', format='ascii.cds').to_pandas()
print('catalogue read in done')
print(van_lane.columns)
print(mega_list.columns)
mega_with_van_lane = mega_list.merge(van_lane[['GaiaDR3', 'Prot']], left_on='gaia_dr3_id', right_on='GaiaDR3').drop_duplicates(subset='gaia_dr3_id')
print(len(mega_with_van_lane))

catalogue read in done
Index(['GaiaDR3', 'Cluster', 'Subgroup', 'DataCatalog', 'ProtSource', 'Prot',
       'ProbHDBScan', 'ProbSource', 'plx', 'RUWE', 'Gmag0', 'BPRP0', 'e_BPRP0',
       'Dustmap', 'Age', 'CMD', 'Dered', 'Phot'],
      dtype='object')
Index(['pl_name', 'hostname', 'gaia_dr3_id', 'gaia_dr2_id', 'tic_id',
       'hd_name', 'ra', 'dec', 'sy_vmag', 'sy_jmag', 'sy_kmag', 'sy_tmag',
       'sy_kepmag', 'sy_gaiamag', 'st_teff', 'st_logg', 'st_met', 'st_mass',
       'st_rad', 'st_spectype', 'st_lum', 'st_age', 'st_ageerr1', 'st_ageerr2',
       'st_rotp', 'pl_orbper', 'pl_rade', 'pl_trandur', 'disc_facility',
       'disc_year', 'mission_source'],
      dtype='object')
17


In [18]:
# ── Start from the full mega_list so ALL stars are preserved ─────────────────
mega_list = pd.read_csv('ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
mega_list['tic_id'] = mega_list['tic_id'].str.replace('TIC ', '', regex=False).astype('Int64')

# ── Pull only the period column from each catalogue, renamed unambiguously ───
prot_jayasinghe = (
    mega_list[['tic_id']].drop_duplicates()
    .merge(jayasinghe[['TIC', 'Per']], left_on='tic_id', right_on='TIC', how='inner')
    [['tic_id', 'Per']]
    .rename(columns={'Per': 'Prot_jayasinghe'})
    .drop_duplicates(subset='tic_id')
)

prot_schochet = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(schochet2025[['dr3_source_id', 'period']], left_on='gaia_dr3_id', right_on='dr3_source_id', how='inner')
    [['gaia_dr3_id', 'period']]
    .rename(columns={'period': 'Prot_schochet'})
    .drop_duplicates(subset='gaia_dr3_id')
)

prot_long = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(long[['Gaia', 'PRot']], left_on='gaia_dr3_id', right_on='Gaia', how='inner')
    [['gaia_dr3_id', 'PRot']]
    .rename(columns={'PRot': 'Prot_long'})
    .drop_duplicates(subset='gaia_dr3_id')
)

prot_van_lane = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(van_lane[['GaiaDR3', 'Prot']], left_on='gaia_dr3_id', right_on='GaiaDR3', how='inner')
    [['gaia_dr3_id', 'Prot']]
    .rename(columns={'Prot': 'Prot_van_lane'})
    .drop_duplicates(subset='gaia_dr3_id')
)

# ── Left-join everything onto mega_list one by one ───────────────────────────
mega_combined = (
    mega_list
    .merge(prot_jayasinghe, on='tic_id',      how='left')
    .merge(prot_schochet,   on='gaia_dr3_id', how='left')
    .merge(prot_long,       on='gaia_dr3_id', how='left')
    .merge(prot_van_lane,   on='gaia_dr3_id', how='left')
)

# ── Summary ───────────────────────────────────────────────────────────────────
prot_cols = ['Prot_jayasinghe', 'Prot_schochet', 'Prot_long', 'Prot_van_lane']
print(f"Total stars: {len(mega_combined)}")
for col in prot_cols:
    print(f"  {col}: {mega_combined[col].notna().sum()} matches")

print(f"\nStars with ANY Prot measurement: {mega_combined[prot_cols].notna().any(axis=1).sum()}")
print(mega_combined[['gaia_dr3_id', 'tic_id'] + prot_cols].head(10))

mega_combined.to_csv('mega_combined.csv', index=False)

NameError: name 'jayasinghe' is not defined

In [17]:
mega_list = pd.read_csv('ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')

gaiadr3 = pd.read_csv('/Users/iansterrett/Downloads/2e7a2502-0ce2-11f1-91f6-bc97e148b76b-O-result.csv')
gaiadr3['source_id'] = gaiadr3['source_id'].astype('Int64')

# Filter for stars that match the mega list AND are labeled SOLAR_LIKE
solar_like_matches = gaiadr3[
    (gaiadr3['source_id'].isin(mega_list['gaia_dr3_id']))
]

print(f"Number of matching SOLAR_LIKE/YSO stars: {len(solar_like_matches)}")
print(solar_like_matches[['source_id', 'best_class_name']])

Number of matching SOLAR_LIKE/YSO stars: 182
                  source_id best_class_name
232     2265406171296997888      SOLAR_LIKE
2342    4664811297844004352      SOLAR_LIKE
3701    2266490873877571712      SOLAR_LIKE
3794    2053454478745972480      SOLAR_LIKE
4161    5094154336332482304      SOLAR_LIKE
...                     ...             ...
280058  2128840912955018368              RS
280070  2130991596358782464      SOLAR_LIKE
284868  1404488390652463872      SOLAR_LIKE
285733  1675923009431370880      SOLAR_LIKE
287285  4031575166693272832      SOLAR_LIKE

[182 rows x 2 columns]


In [2]:
# ── Load all catalogues ───────────────────────────────────────────────────────
mega_list = pd.read_csv('ASTR502_Mega_Target_List.csv')
mega_list['gaia_dr3_id'] = mega_list['gaia_dr3_id'].str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
mega_list['tic_id'] = mega_list['tic_id'].str.replace('TIC ', '', regex=False).astype('Int64')

jayasinghe = Table.read('/Users/iansterrett/Downloads/jayasinghe2018/catv2021.dat',
                         readme='https://cdsarc.cds.unistra.fr/ftp/II/366/ReadMe',
                         format='ascii.cds').to_pandas()

print('cat 1 loaded')

van_lane = Table.read('https://content.cld.iop.org/journals/0004-637X/986/1/59/revision1/apjadcd73t1_mrt.txt',
                       format='ascii.cds').to_pandas()

print('cat 2  loaded')

long = Table.read('/Users/iansterrett/Downloads/long2023/table5.dat', readme='https://cdsarc.cds.unistra.fr/ftp/J/ApJS/268/30/ReadMe', format='ascii.cds').to_pandas()

print('cat 3 loaded')

schochet2025 = pd.read_csv('/Users/iansterrett/Downloads/reportable.csv')
print(schochet2025.columns)

print('cat 4 loaded')


cat 1 loaded
cat 2  loaded
cat 3 loaded
Index(['Unnamed: 0', 'asas_sn_id', 'period', 'sigma', 'dr3_source_id',
       'dr2_source_id', 'edr3_source_id', 'KIC', 'tic_id', 'ra_rad', 'dec_rad',
       'parallax', 'abs_g_mag', 'Tmag', 'phot_g_mean_mag', 'phot_bp_mean_mag',
       'phot_rp_mean_mag', 'radius_val', 'lum_val', 'ruwe', 'vbroad',
       'vbroad_error', 'catwise_w1', 'catwise_w2', 'mh_xgboost',
       'teff_xgboost', 'logg_xgboost', 'in_training', 'in_xgboost_training',
       'simulation_number'],
      dtype='object')
cat 4 loaded


In [3]:
# ── Build per-catalogue match tables ─────────────────────────────────────────
prot_jayasinghe = (
    mega_list[['tic_id']].drop_duplicates()
    .merge(jayasinghe[['TIC', 'Per']], left_on='tic_id', right_on='TIC', how='inner')
    [['tic_id', 'Per']]
    .rename(columns={'Per': 'Prot_jayasinghe'})
    .drop_duplicates(subset='tic_id')
)

prot_van_lane = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(van_lane[['GaiaDR3', 'Prot']], left_on='gaia_dr3_id', right_on='GaiaDR3', how='inner')
    [['gaia_dr3_id', 'Prot']]
    .rename(columns={'Prot': 'Prot_van_lane'})
    .drop_duplicates(subset='gaia_dr3_id')
)

prot_schochet = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(schochet2025[['GaiaDR3', 'period']], left_on='gaia_dr3_id', right_on='dr3_source_id', how='inner')
    [['gaia_dr3_id', 'period']]
    .rename(columns={'Prot': 'Prot_van_lane'})
    .drop_duplicates(subset='gaia_dr3_id')
)

prot_long = (
    mega_list[['gaia_dr3_id']].drop_duplicates()
    .merge(long[['GaiaDR3', 'PRot']], left_on='gaia_dr3_id', right_on='GaiaDR3', how='inner')
    [['gaia_dr3_id', 'PRot']]
    .rename(columns={'Prot': 'Prot_van_lane'})
    .drop_duplicates(subset='gaia_dr3_id')
)

KeyError: "['GaiaDR3'] not in index"

In [ ]:
# ── Left-join everything onto mega_list ───────────────────────────────────────
mega_combined = (
    mega_list
    .merge(prot_jayasinghe, on='tic_id',      how='left')
    .merge(prot_van_lane,   on='gaia_dr3_id', how='left')
     .merge(prot_schochet, on='gaia_dr3_id', how='left')
     .merge(prot_long,     on='gaia_dr3_id', how='left')
)

# ── Summary stats ─────────────────────────────────────────────────────────────
prot_cols = ['Prot_jayasinghe', 'Prot_van_lane','Prot_schochet', 'Prot_long']  # add others as you load them

print(f"Total stars in master table: {len(mega_combined)}")
print()

# Matches per catalogue
for col in prot_cols:
    print(f"  {col}: {mega_combined[col].notna().sum()} matches")

# Stars with at least one match across all catalogues
any_match = mega_combined[prot_cols].notna().any(axis=1)
print(f"\nStars with ANY Prot measurement: {any_match.sum()}")

# Duplicate check
dup_gaia = mega_combined[mega_combined.duplicated(subset='gaia_dr3_id', keep=False)]
dup_tic  = mega_combined[mega_combined.duplicated(subset='tic_id', keep=False)]
print(f"\nDuplicate gaia_dr3_id rows: {len(dup_gaia)}")
print(f"Duplicate tic_id rows:      {len(dup_tic)}")

if len(dup_gaia) > 0:
    print("\nDuplicated gaia_dr3_id entries:")
    print(dup_gaia[['gaia_dr3_id', 'tic_id'] + prot_cols])

# Stars matched in multiple catalogues simultaneously
multi_match = mega_combined[prot_cols].notna().sum(axis=1)
print(f"\nStars matched in 2+ catalogues: {(multi_match >= 2).sum()}")

# Save
mega_combined.to_csv('mega_combined.csv', index=False)
print("\nSaved to mega_combined.csv")